# ArrayInversionCount

[Codility task](https://app.codility.com/programmers/trainings/4/array_inversion_count/) - Exercise 4, Algorithmic skills

An **inversion** is a pair `(P, Q)` with `P < Q` and `A[Q] < A[P]`. `solution(A)` returns the number of inversions, or `-1` if it exceeds `1,000,000,000`.

Example: `[-1, 6, 3, 4, 7, 4]` has four inversions, `(1,2) (1,3) (1,5) (4,5)`.

Assumptions: `N` in `[0..100,000]`, 32-bit signed values.

## Idea: counting while merge-sorting

When merging two sorted halves, if the next element comes from the **right** half, it is smaller than every element still waiting in the left half: each of those is an inversion with it.

- ties: take the **left** element first (equal elements are not inversions);
- add `len(left) - i` to the counter each time a right element is taken.

Time O(N log N), space O(N). A second implementation with a **Fenwick tree** (binary indexed tree) cross-checks the result.

In [ ]:
LIMIT = 1_000_000_000


def count_inversions(A):
    """Exact number of inversions via bottom-up merge sort."""
    items = list(A)
    n = len(items)
    buffer = items[:]
    inversions = 0
    width = 1
    while width < n:
        for lo in range(0, n, 2 * width):
            mid = min(lo + width, n)
            hi = min(lo + 2 * width, n)
            i, j, k = lo, mid, lo
            while i < mid and j < hi:
                if items[j] < items[i]:                 # strict: equal values are not inversions
                    buffer[k] = items[j]
                    j += 1
                    inversions += mid - i               # everything left in the left half is greater
                else:
                    buffer[k] = items[i]
                    i += 1
                k += 1
            while i < mid:
                buffer[k] = items[i]
                i += 1
                k += 1
            while j < hi:
                buffer[k] = items[j]
                j += 1
                k += 1
        items, buffer = buffer, items
        width *= 2
    return inversions


def solution(A):
    total = count_inversions(A)
    return -1 if total > LIMIT else total

## Alternative: Fenwick tree

Compress the values to ranks `1..m`. Scan left to right: for each element, the number of earlier elements that are **greater** is `processed - (number of earlier elements <= it)`.

In [ ]:
def count_inversions_bit(A):
    ranks = {v: r for r, v in enumerate(sorted(set(A)), start=1)}
    tree = [0] * (len(ranks) + 1)

    def add(i):
        while i < len(tree):
            tree[i] += 1
            i += i & -i

    def prefix(i):
        s = 0
        while i > 0:
            s += tree[i]
            i -= i & -i
        return s

    total = 0
    for processed, value in enumerate(A):
        r = ranks[value]
        total += processed - prefix(r)          # earlier elements strictly greater than value
        add(r)
    return total

## Official example

In [ ]:
A = [-1, 6, 3, 4, 7, 4]
assert solution(A) == 4
assert count_inversions_bit(A) == 4
print("example OK")

## Edge cases

In [ ]:
assert solution([]) == 0                                # N = 0
assert solution([5]) == 0
assert solution([1, 2, 3, 4]) == 0                      # sorted
assert solution([4, 3, 2, 1]) == 6                      # reversed: N(N-1)/2
assert solution([2, 2, 2]) == 0                         # equal elements: no inversions
assert solution([2, 1, 2, 1]) == 3
assert solution([-2**31, 2**31 - 1]) == 0               # extreme values, in order
assert solution([2**31 - 1, -2**31]) == 1               # extreme values, inverted
print("edge cases OK")

## Brute-force check

In [ ]:
import random


def brute(A):
    return sum(1 for p in range(len(A)) for q in range(p + 1, len(A)) if A[q] < A[p])


rng = random.Random(2)
for _ in range(3000):
    n = rng.randint(0, 30)
    A = [rng.randint(-5, 5) for _ in range(n)]
    expected = brute(A)
    assert count_inversions(A) == expected, A
    assert count_inversions_bit(A) == expected, A
print("merge sort and Fenwick tree match brute force on 3000 random arrays")

## The limit of 1,000,000,000

A reversed array of length `n` has `n(n-1)/2` inversions. We find the largest `n` that stays within the limit and test both sides of the boundary.

In [ ]:
from math import isqrt

n = (1 + isqrt(1 + 8 * LIMIT)) // 2
while n * (n - 1) // 2 > LIMIT:
    n -= 1
print(f"largest reversed array within the limit: n = {n} with {n * (n - 1) // 2:,} inversions")

assert solution(list(range(n, 0, -1))) == n * (n - 1) // 2          # at most 10^9: reported exactly
assert solution(list(range(n + 1, 0, -1))) == -1                    # one more element: over the limit
assert solution(list(range(100_000, 0, -1))) == -1                  # worst case of the statement
print("limit OK")

## Performance

In [ ]:
import time

N = 100_000
cases = {
    "random": [rng.randint(-2**31, 2**31 - 1) for _ in range(N)],
    "sorted": list(range(N)),
    "reversed": list(range(N, 0, -1)),
    "many ties": [rng.randint(0, 3) for _ in range(N)],
}
for name, A in cases.items():
    start = time.perf_counter()
    result = solution(A)
    print(f"{name:10s} -> {result:>14,} in {time.perf_counter() - start:.3f}s")
assert count_inversions(cases["random"]) == count_inversions_bit(cases["random"])

## Notes

- Inversions measure how far an array is from sorted (a bubble sort needs exactly that many swaps).
- Merge sort and a Fenwick tree are the two classic O(N log N) approaches.
- In languages with fixed-size integers, accumulate the count in a 64-bit type: it can reach about 5 * 10^9.